# Домашнее задание 6

В этом задании мы:
1. Посмотрим на то, как получать эмбеддинги слов с помощью счетчиков и с помощью word2vec.
2. Построим простую сеть для классификации текста, замерим качество.
3. Посмотрим, как меняется качество этой сети в зависимости от качества эмбеддингов.
4. Построим TextCNN, обучим, сравним качество.

Работать будем с датасетом IMDb.

In [5]:
# Загрузка данных IMDb с помощью Keras
from keras.datasets import imdb

# Датасет хранится в двух кусках: сами данные и индексы (трейн/тест)
# Скачаем индексы
(X_train_indices, y_train), (X_test_indices, y_test) = imdb.load_data(num_words=10000)
# И сами данные. Это словарь вида {"слово": 0, "другое": 100}
# Выделим первые три слота под специальные токены: <PAD>, <START> и <UNK>
# Нумерация в исходном датасете идет с единицы, поэтому вычтем 1
word_to_index = {k: v - 1 + 3 for k, v in imdb.get_word_index().items()} | {
    "<PAD>": 0,
    "<START>": 1,
    "<UNK>": 2,
}
index_to_word = {index: word for word, index in word_to_index.items()}


# Функция для преобразования последовательности индексов в последовательность слов
def indices_to_words(indices):
    return " ".join(index_to_word.get(index, "<UNK>") for index in indices)


# Список строк, вида ["Good movie", "Bad film", ...]
X_train = [indices_to_words(indices) for indices in X_train_indices]
X_test = [indices_to_words(indices) for indices in X_test_indices]

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


В отзывах на фильмы скорее всего будут использовать различные сокращения фраз.
Например, AFAIK - "as far as I know".
Мы собрали словарь некоторых из них.

In [6]:
chat_words = {
    "AFAIK": "As Far As I Know",
    "AFK": "Away From Keyboard",
    "ASAP": "As Soon As Possible",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "A3": "Anytime, Anywhere, Anyplace",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRT": "Be Right There",
    "BTW": "By The Way",
    "B4": "Before",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FWIW": "For What It's Worth",
    "FYI": "For Your Information",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GN": "Good Night",
    "GMTA": "Great Minds Think Alike",
    "GR8": "Great!",
    "G9": "Genius",
    "IC": "I See",
    "ICQ": "I Seek you (also a chat program)",
    "ILU": "I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "LDR": "Long Distance Relationship",
    "LTNS": "Long Time No See",
    "L8R": "Later",
    "MTE": "My Thoughts Exactly",
    "M8": "Mate",
    "NRN": "No Reply Necessary",
    "OIC": "Oh I See",
    "PITA": "Pain In The A..",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "QPSA": "Que Pasa?",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A.. Off",
    "SK8": "Skate",
    "STATS": "Your sex and age",
    "ASL": "Age, Sex, Location",
    "THX": "Thank You",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "WB": "Welcome Back",
    "WTF": "What The F...",
    "WTG": "Way To Go!",
    "WUF": "Where Are You From?",
    "W8": "Wait...",
    "7K": "Sick:-D Laughter",
    "TFW": "That feeling when",
    "MFW": "My face when",
    "MRW": "My reaction when",
    "IFYP": "I feel your pain",
    "LOL": "Laughing out loud",
    "TNTL": "Trying not to laugh",
    "JK": "Just kidding",
    "IDC": "I don’t care",
    "ILY": "I love you",
    "IMU": "I miss you",
    "ADIH": "Another day in hell",
    "ZZZ": "Sleeping, bored, tired",
    "WYWH": "Wish you were here",
    "BAE": "Before anyone else",
    "FIMH": "Forever in my heart",
    "BSAAW": "Big smile and a wink",
    "BWL": "Bursting with laughter",
    "LMAO": "Laughing my a** off",
    "BFF": "Best friends forever",
    "CSL": "Can’t stop laughing",
}
# Чтобы не иметь проблем с регистром, приведем все к нижнему регистру
chat_words = {key.lower(): value.lower() for key, value in chat_words.items()}

### Задание №1
Посчитайте, какое количество таких сокращений встречается в тестовом наборе данных `X_train`.

Например, для строк:
```
[
    "AFAIK, he is AFK",
    "Need to go, BBS",
    "Are you AFK ?",
]
```
ответ будет 4, т.к. всего в текстах встретилось 4 сокращения: AFAIK, AFK, BBS, AFK.

Сдайте в ЛМС одно число - количество сокращений. Например, `123`.

In [20]:
type(X_train_indices)

numpy.ndarray

In [10]:
indices_to_words(X_train_indices[0])

"<START> that on as about parts admit ready speaking really care boot see holy and again who each a are any about brought life what power <UNK> br they sound everything a though and part life look <UNK> fan recommend like and part elegant successful for feeling from this based and take what as of those core movie that on and manage airplane 4 and on me because i as about parts from been was this military and on for kill for i as cinematography with <UNK> a which let i is left is two a and seat raises as sound see worried by and still i as from running a are off good who scene some are church by of on i come he bad more a that gives as into <UNK> is and films best commenting was each and <UNK> to rid a beyond who me about parts final his keep special has to and <UNK> manages this characters how and perhaps was american too at references no his something of enough russ with and bit on film say final his sound a back one jews with good who he there's made are characters and bit really as 

In [13]:
chat_words.keys()

dict_keys(['afaik', 'afk', 'asap', 'atk', 'atm', 'a3', 'bak', 'bbl', 'bbs', 'bfn', 'b4n', 'brb', 'brt', 'btw', 'b4', 'cu', 'cul8r', 'cya', 'faq', 'fc', 'fwiw', 'fyi', 'gal', 'gg', 'gn', 'gmta', 'gr8', 'g9', 'ic', 'icq', 'ilu', 'imho', 'imo', 'iow', 'irl', 'ldr', 'ltns', 'l8r', 'mte', 'm8', 'nrn', 'oic', 'pita', 'prt', 'prw', 'qpsa', 'rofl', 'roflol', 'rotflmao', 'sk8', 'stats', 'asl', 'thx', 'ttfn', 'ttyl', 'u2', 'u4e', 'wb', 'wtf', 'wtg', 'wuf', 'w8', '7k', 'tfw', 'mfw', 'mrw', 'ifyp', 'lol', 'tntl', 'jk', 'idc', 'ily', 'imu', 'adih', 'zzz', 'wywh', 'bae', 'fimh', 'bsaaw', 'bwl', 'lmao', 'bff', 'csl'])

In [15]:
y_train.shape

(25000,)

In [17]:
len(index_to_word.keys())

88587

In [ ]:
from collections import Counter
import numpy as np

flat_words = [index_to_word.get(index) for index in np.concatenate(X_train_indices)]
print(len(flat_words))

# 1. Считаем частоту всех слов во втором списке
target_counter = Counter(flat_words)
print(target_counter)

# 2. Создаем словарь с результатами
result = {}
for word in chat_words.keys():
    result[word] = target_counter.get(word, 0) # get вернет 0, если слова нет

print(result)
print(sum(result.values()))

5967841
Counter({'<UNK>': 341611, 'and': 336148, 'a': 164097, 'of': 163040, 'to': 145847, 'is': 135708, 'br': 107313, 'in': 101871, 'it': 93934, 'i': 79058, 'this': 77142, 'that': 75974, 'was': 69787, 'as': 48195, 'for': 46927, 'with': 44335, 'movie': 44122, 'but': 43564, 'film': 42594, 'on': 39095, 'not': 34188, 'you': 30610, 'are': 29877, 'his': 29425, 'have': 29366, 'he': 27726, 'be': 26952, 'one': 26948, 'all': 26513, '<START>': 25000, 'at': 23953, 'by': 23507, 'an': 22539, 'they': 21538, 'who': 21139, 'so': 20599, 'from': 20586, 'like': 20494, 'her': 20272, 'or': 18407, 'just': 17994, 'about': 17759, "it's": 17371, 'out': 17153, 'has': 17092, 'some': 16790, 'if': 16790, 'there': 15743, 'what': 15735, 'good': 15349, 'more': 15100, 'when': 14246, 'very': 14175, 'up': 14062, 'no': 13274, 'time': 12690, 'she': 12682, 'even': 12657, 'my': 12650, 'would': 12492, 'which': 12238, 'only': 12041, 'story': 11915, 'really': 11892, 'see': 11734, 'their': 11460, 'had': 11376, 'can': 11287, 'wer

In [27]:
# решение авторов
import tqdm

cnt = 0
for text in tqdm.tqdm(X_train):
    for word in text.split():
        if word in chat_words:
            cnt += 1
cnt

100%|██████████| 25000/25000 [00:01<00:00, 23318.39it/s]


475

Подумайте самостоятельно: стоит ли заменять эти сокращения в `X_train` и `X_test`?

## Простая модель, простой эмбеддинг
Обучим на этих данных классификатор.
Для этого надо слова превратить в вектора - _эмбеддинги_, затем построить модель.

### Задание №2

Обучите `CountVectorizer` из scikit-learn на текстах - это будет модель для эмбеддингов.
Используйте `max_features=10000`.

Затем обучите простую нейросеть (код ниже) на классификацию отзывов.

Сдайте в ЛМС код модели `SimpleNeuralNet` и файл с весами обученной модели (`model.pt`).

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


class SimpleNeuralNet(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        out = self.fc(x)
        return out.squeeze()


vectorizer = ...
X_train_bow = ...
X_test_bow = ...

In [6]:
# Почистим память перед дальнейшими экспериментами
del X_train_bow, X_test_bow, vectorizer

## Word2Vec

Попробуем использовать более продвинутую модель для подсчета эмбеддингов.
Возьмем `word2vec` - это популярная модель для эмбеддингов, ее часто используют в качестве бейзлайна.

In [7]:
from gensim.downloader import load

word2vec = load("word2vec-google-news-300")
word2vec.similar_by_word("cat")

Про word2vec часто говорят, что он дает "осмысленные" эмбеддинги.
Давайте это проверим на примере.

### Задание №3
Какое слово получается, если сделать Paris - France + Germany, и с какой косинусной близостью?

Сдайте в ЛМС слово и близость в формате `'Munich', 0.98`.

Можно надеяться, что если эмбеддинги будут более осмысленные, то качество модели увеличится.
Давайте проверять.

### Задание №4

Возьмите ту же модель `SimpleNeuralNet` и обучите её, взяв эмбеддинги из `word2vec`.

Вы, возможно, подумаете: `word2vec` выдает по вектору на каждое слово, а в предложении слов много.
Как с этим быть, предлагаем подумать самостоятельно.

Сдайте в ЛМС код `SimpleNeuralNet` и файл с весами модели `model.pt`.

In [9]:
import numpy as np


def sentence_to_embedding(sentence: str) -> np.ndarray:
    # На каждое предложение возвращаем 1 вектор размерности (300, )
    # Подумайте, что делать со словами, которых нет в словаре word2vec.
    ...


X_train_w2v = np.stack(
    [sentence_to_embedding(sentence) for sentence in tqdm.tqdm(X_train)]
)
X_test_w2v = np.stack(
    [sentence_to_embedding(sentence) for sentence in tqdm.tqdm(X_test)]
)
assert X_train_w2v.shape == (25_000, 300)
assert X_test_w2v.shape == (25_000, 300)

In [10]:
...
torch.save(model.state_dict(), "model-w2v.pt")

In [12]:
del X_train_w2v, X_test_w2v

Возможно, качество вырастет не так сильно, но зато теперь мы можем считать эмбеддинги отдельно для каждого слова.
А это значит, что мы можем попробовать учесть порядок слов в предложении.

## TextCNN

Модель TextCNN - одна из первых попыток учесть положение слов в тексте.
Попробуем обучить эту модель.

### Задание №5

Обучите `TextCNN` на датасете IMDb, используя эмбеддинги `word2vec` и паддинг из прошлого пункта.

Не забудьте, что в таком подходе каждое предложение имеет размерность не `(300, )`, а `(n_words, 300)`.
А это значит:
- нужен паддинг;
- нужно переписать логику подсчета эмбеддингов для предложения.

Вы можете поискать подсказки в семинаре.

Сдайте в ЛМС код `TextCNN` и файл с весами модели.

### Задание №6
Поэкспериментируйте с моделью и данными, попробуйте выбить наибольшее качество.

Сдайте в ЛМС наибольший accuracy, который у вас получился.